<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/FinalModelSyntheticData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Final Model with Synthetic Data")

Final Model with Synthetic Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, EfficientNetB2
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from datasets import load_dataset, Dataset

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import numpy as np

from PIL import Image
import os
import matplotlib.pyplot as plt
import cv2
import torch
import glob
import math

from keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Permute, Concatenate, Conv2D, Add, Activation, Lambda
from tensorflow.keras import backend as K

!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-612034a0-d35f-acef-ca2a-10f8bd2985cf)


In [ ]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-c08a401c53fe53(…):   0%|          | 0.00/22.6M [00:00<?, ?B/s]

data/test-00000-of-00001-44110b9df98c558(…):   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1280 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5120
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1280
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented'])}


In [ ]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

synth_dir = "/content/drive/MyDrive/Moderate_Demented_Synthetic"
synth_files = sorted(glob.glob(f"{synth_dir}/seed*.png"))
print("Found", len(synth_files), "synthetic images.")

def load_synth_example(path):
    img = Image.open(path).convert("L")
    img = np.array(img)
    return {"image": img, "label": 1}

synth_data = [load_synth_example(p) for p in synth_files]
synth_dataset = Dataset.from_list(synth_data)
print(synth_dataset)

def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)


def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    def keep_images_rgb():
        return images

    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_images_rgb)
    return images

def to_tensorflow_dataset(dataset_split, image_size, preprocess_fn, shuffle=False):
        dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )
        def preprocess_input(example_dict):
            images = tf.cast(example_dict["image"], tf.float32)
            images = ensure_channel_dim(images)
            images = ensure_rgb_channels(images)
            images.set_shape([None, None, 3])
            images = tf.image.resize(images, (image_size, image_size))

            images = preprocess_fn(images)

            labels = tf.cast(example_dict["label"], tf.int32)
            labels = tf.one_hot(labels, depth=4)
            return images, labels

        dataset_tf = dataset_tf.map(preprocess_input,
                                    num_parallel_calls=tf.data.AUTOTUNE)
        dataset_tf = dataset_tf.batch(50)
        return dataset_tf.prefetch(tf.data.AUTOTUNE)

vgg_clean_train = to_tensorflow_dataset(train, 224, vgg_preprocess, shuffle=True)
eff_clean_train = to_tensorflow_dataset(train, 260, eff_preprocess, shuffle=True)

vgg_synth_train = to_tensorflow_dataset(synth_dataset, 224, vgg_preprocess, shuffle=True)
eff_synth_train = to_tensorflow_dataset(synth_dataset, 260, eff_preprocess, shuffle=True)

full_vgg_synth_train = vgg_synth_train.concatenate(vgg_clean_train)
full_eff_synth_train = eff_synth_train.concatenate(eff_clean_train)

vgg_train_dataset = to_tensorflow_dataset(train, 224, vgg_preprocess, shuffle=True)
vgg_val_dataset = to_tensorflow_dataset(val, 224, vgg_preprocess, shuffle=False)
vgg_test_dataset = to_tensorflow_dataset(test, 224, vgg_preprocess, shuffle=False)

eff_train_dataset = to_tensorflow_dataset(train, 260, eff_preprocess, shuffle=True)
eff_val_dataset = to_tensorflow_dataset(val, 260, eff_preprocess, shuffle=False)
eff_test_dataset = to_tensorflow_dataset(test, 260, eff_preprocess, shuffle=False)

vgg_full_train_dataset = vgg_train_dataset
eff_full_train_dataset = eff_train_dataset


Found 50 synthetic images.
Dataset({
    features: ['image', 'label'],
    num_rows: 50
})


In [ ]:
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)

vgg_base.trainable = False

vgg_inputs = tf.keras.Input(shape=(224, 224, 3))
x = vgg_base(vgg_inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
vgg_outputs = layers.Dense(4, activation="softmax")(x)

vgg_model = tf.keras.Model(vgg_inputs, vgg_outputs)

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

vgg_model.summary()

# use add_1 for gradcam

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,044 (56.64 MB)

 Trainable params: 132,356 (517.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
VGG_EPOCHS = 50

class_weight = {
    0: 1.25,
    1: 2.0,
    2: 1.0,
    3: 1.0,
}

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

vgg_history_phase0 = vgg_model.fit(
    vgg_clean_train,
    validation_data=vgg_val_dataset,
    epochs=1,
    class_weight=class_weight,
)

vgg_history_phase1 = vgg_model.fit(
    full_vgg_synth_train,
    validation_data=vgg_val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop]
)

vgg_history_phase2 = vgg_model.fit(
    vgg_full_train_dataset,
    validation_data = vgg_val_dataset,
    epochs = VGG_EPOCHS,
    class_weight = class_weight,
    callbacks = [reduce_lr, early_stop]
)

print("\n=== PHASE 2: Fine-tuning top layers ===")
vgg_base.trainable = True
for layer in vgg_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in vgg_base.layers])} / {len(vgg_base.layers)}")

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

vgg_history_phase3 = vgg_model.fit(
    vgg_full_train_dataset,
    validation_data = vgg_val_dataset,
    epochs = VGG_EPOCHS,
    class_weight = class_weight,
    callbacks = [reduce_lr, early_stop]
)

print("\n=== FINAL VGG16 EVALUATION ===")
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(vgg_test_dataset, verbose=0)
print(f"VGG16 final test accuracy: {vgg_test_acc:.4f}")


82/82 ━━━━━━━━━━━━━━━━━━━━ 60s 493ms/step - accuracy: 0.4426 - loss: 2.8891 - val_accuracy: 0.5664 - val_loss: 0.9279
Epoch 1/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 196ms/step - accuracy: 0.5077 - loss: 1.4759 - val_accuracy: 0.5684 - val_loss: 0.9095 - learning_rate: 0.0010
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 196ms/step - accuracy: 0.5571 - loss: 1.1035 - val_accuracy: 0.6035 - val_loss: 0.8780 - learning_rate: 0.0010
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 196ms/step - accuracy: 0.5731 - loss: 1.0324 - val_accuracy: 0.5977 - val_loss: 0.8568 - learning_rate: 0.0010
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.6257 - loss: 0.8648 - val_accuracy: 0.6230 - val_loss: 0.8173 - learning_rate: 0.0010
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 198ms/step - accuracy: 0.6585 - loss: 0.8197 - val_accuracy: 0.6494 - val_loss: 0.8168 - learning_rate: 0.0010
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 16s 196ms/step - accuracy: 0.6484 - loss: 0.8036 - val_accuracy: 0.6357 - val_l

In [ ]:
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

vgg_path = os.path.join(save_dir, "SYNTH_VGG.keras")
vgg_model.save(vgg_path)

print("Saved VGG model to:", vgg_path)

Saved VGG model to: /content/drive/MyDrive/alz_models/SYNTH_VGG.keras


In [ ]:
y_true = []
y_pred = []

for images, labels in vgg_test_dataset:
    preds = vgg_model.predict(images)

    true_classes = np.argmax(labels.numpy(), axis=1)

    y_true.extend(true_classes)
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 18s 7s/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
[[155   0   6  

In [ ]:
eff_base = EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(260, 260, 3),
)
eff_base.trainable = False

eff_inputs = tf.keras.Input(shape=(260, 260, 3))
y = eff_base(eff_inputs, training=False)
y = layers.GlobalAveragePooling2D()(y)
y = layers.Dense(256, activation="relu")(y)
y = layers.Dropout(0.5)(y)
eff_outputs = layers.Dense(4, activation="softmax")(y)

eff_model = tf.keras.Model(eff_inputs, eff_outputs)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

eff_model.summary()

# use top_conv for gradcam

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,130,301 (31.01 MB)

 Trainable params: 361,732 (1.38 MB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [ ]:
EFF_EPOCHS = 50

class_weight = {
    0: 1.25,
    1: 2.0,
    2: 1.0,
    3: 1.0,
}

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

eff_history_phase0 = eff_model.fit(
    eff_clean_train,
    validation_data=eff_val_dataset,
    epochs=1,
    class_weight=class_weight,
)

eff_history_phase1 = eff_model.fit(
    full_eff_synth_train,
    validation_data=eff_val_dataset,
    epochs=10,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop]
)

eff_history_phase2 = eff_model.fit(
    eff_full_train_dataset,
    validation_data=eff_val_dataset,
    epochs=EFF_EPOCHS,
    class_weight=class_weight,
    callbacks = [reduce_lr, early_stop]
)


print("\n=== PHASE 2: Fine-tuning top layers ===")
eff_base.trainable = True
for layer in eff_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in eff_base.layers])} / {len(eff_base.layers)}")

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

eff_history_phase3 = eff_model.fit(
    eff_full_train_dataset,
    validation_data=eff_val_dataset,
    epochs=EFF_EPOCHS,
    class_weight=class_weight,
    callbacks=[reduce_lr, early_stop],
)

print("\n=== FINAL EfficientNetB2 EVALUATION ===")
eff_test_loss, eff_test_acc = eff_model.evaluate(eff_test_dataset, verbose=0)
print(f"EfficientNetB2 final test accuracy: {eff_test_acc:.4f}")

82/82 ━━━━━━━━━━━━━━━━━━━━ 122s 902ms/step - accuracy: 0.4584 - loss: 1.1855 - val_accuracy: 0.5459 - val_loss: 0.9416
Epoch 1/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 171ms/step - accuracy: 0.5014 - loss: 1.3936 - val_accuracy: 0.5430 - val_loss: 0.9088 - learning_rate: 0.0010
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 169ms/step - accuracy: 0.5267 - loss: 1.1537 - val_accuracy: 0.5703 - val_loss: 0.8931 - learning_rate: 0.0010
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 167ms/step - accuracy: 0.5698 - loss: 1.0454 - val_accuracy: 0.5762 - val_loss: 0.8734 - learning_rate: 0.0010
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 167ms/step - accuracy: 0.5922 - loss: 0.9832 - val_accuracy: 0.5879 - val_loss: 0.8656 - learning_rate: 0.0010
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 167ms/step - accuracy: 0.5975 - loss: 0.9176 - val_accuracy: 0.5840 - val_loss: 0.8539 - learning_rate: 0.0010
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 168ms/step - accuracy: 0.6049 - loss: 0.8962 - val_accuracy: 0.5938 - val_

In [ ]:
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

eff_path = os.path.join(save_dir, "SYNTH_EFF.keras")
eff_model.save(eff_path)

print("Saved EFF model to:", eff_path)

Saved EFF model to: /content/drive/MyDrive/alz_models/SYNTH_EFF.keras


In [ ]:
y_true = []
y_pred = []

for images, labels in eff_test_dataset:
    preds = eff_model.predict(images)

    true_classes = np.argmax(labels.numpy(), axis=1)

    y_true.extend(true_classes)
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
vgg_probs = vgg_model.predict(vgg_test_dataset)
eff_probs = eff_model.predict(eff_test_dataset)

ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

onehot_labels = np.concatenate([y.numpy() for _, y in vgg_test_dataset], axis=0)
y_true = np.argmax(onehot_labels, axis=1)

vgg_preds = np.argmax(vgg_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import drive
import os

vgg_path = "/content/drive/MyDrive/alz_models/SYNTH_VGG.keras"
eff_path = "/content/drive/MyDrive/alz_models/SYNTH_EFF.keras"

vggmodel = load_model(vgg_path, compile=False)
effmodel = load_model(eff_path, compile=False)

vgg_probs = vggmodel.predict(vgg_test_dataset)
eff_probs = effmodel.predict(eff_test_dataset)

ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

onehot_labels = np.concatenate([y.numpy() for _, y in vgg_test_dataset], axis=0)
y_true = np.argmax(onehot_labels, axis=1)

vgg_preds = np.argmax(vgg_probs, axis=1)
eff_preds = np.argmax(eff_probs, axis=1)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

26/26 ━━━━━━━━━━━━━━━━━━━━ 27s 526ms/step
26/26 ━━━━━━━━━━━━━━━━━━━━ 22s 492ms/step
VGG16 test accuracy:  0.96171875
EfficientNetB2 test acc:  0.8890625
Ensemble test accuracy:  0.9703125

Ensemble classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9128    0.9544       172
           1     1.0000    0.9333    0.9655        15
           2     0.9736    0.9874    0.9804       634
           3     0.9549    0.9695    0.9622       459

    accuracy                         0.9703      1280
   macro avg     0.9821    0.9508    0.9656      1280
weighted avg     0.9707    0.9703    0.9702      1280


Ensemble confusion matrix:
[[157   0   3  12]
 [  0  14   0   1]
 [  0   0 626   8]
 [  0   0  14 445]]
